# 04 — Topic Landscape (V1-S06)

Renders the BERTopic v1 artifacts: intertopic distance map, temporal share heatmap, per-journal-per-year stacked areas, and a 20-topic spot-check scaffold for Gate G1.

Inputs: `data/v1/topics.parquet`, `data/v1/topic_hierarchy.parquet`, `data/v1/topic_sweep.parquet`, `models/v1/bertopic_v1/`, `data/v1/papers.duckdb` (`papers_distinct` view).

## 1. Setup

In [1]:
from __future__ import annotations

import os
from pathlib import Path

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import matplotlib

matplotlib.use("Agg")
import duckdb
import matplotlib.pyplot as plt
import pandas as pd

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent

DUCKDB_PATH = REPO_ROOT / "data/v1/papers.duckdb"
TOPICS_PATH = REPO_ROOT / "data/v1/topics.parquet"
HIERARCHY_PATH = REPO_ROOT / "data/v1/topic_hierarchy.parquet"
SWEEP_PATH = REPO_ROOT / "data/v1/topic_sweep.parquet"
MODEL_DIR = REPO_ROOT / "models/v1/bertopic_v1"
FIGURES_DIR = REPO_ROOT / "docs/figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
GATES_DIR = REPO_ROOT / "docs/gates"
GATES_DIR.mkdir(parents=True, exist_ok=True)
LANDSCAPE_HTML = FIGURES_DIR / "topic_landscape.html"
SHARE_BY_YEAR_PNG = FIGURES_DIR / "topic_share_by_year.png"
SHARE_BY_JOURNAL_YEAR_PNG = FIGURES_DIR / "topic_share_by_journal_year.png"
SPOTCHECK_CSV = GATES_DIR / "G1_spotcheck.csv"

for p in [DUCKDB_PATH, TOPICS_PATH, HIERARCHY_PATH, SWEEP_PATH, MODEL_DIR]:
    print(f"{p.exists()!s:5s}  {p}")

True   /Users/samersalman/Desktop/SciField/data/v1/papers.duckdb
True   /Users/samersalman/Desktop/SciField/data/v1/topics.parquet
True   /Users/samersalman/Desktop/SciField/data/v1/topic_hierarchy.parquet
True   /Users/samersalman/Desktop/SciField/data/v1/topic_sweep.parquet
True   /Users/samersalman/Desktop/SciField/models/v1/bertopic_v1


## 2. Read artifacts and join to `papers_distinct`

In [2]:
topics = pd.read_parquet(TOPICS_PATH)
hier = pd.read_parquet(HIERARCHY_PATH)
sweep_df = pd.read_parquet(SWEEP_PATH)

print(f"topics rows: {len(topics):,}; unique topic_id: {topics['topic_id'].nunique()}")
print(f"hierarchy rows: {len(hier):,}")
print(f"sweep shape: {sweep_df.shape}")

# Ensure papers_distinct view exists; create it via the public helper if not.
_probe = duckdb.connect(str(DUCKDB_PATH), read_only=True)
_views = {row[0] for row in _probe.execute("SHOW TABLES").fetchall()}
_probe.close()
if "papers_distinct" not in _views:
    from scifield.thematic.dedup import ensure_papers_distinct_view

    con_rw = duckdb.connect(str(DUCKDB_PATH))
    ensure_papers_distinct_view(con_rw)
    con_rw.close()

con = duckdb.connect(str(DUCKDB_PATH), read_only=True)
papers = con.execute(
    "SELECT CAST(pmid AS BIGINT) AS pmid, title, journal, year FROM papers_distinct"
).fetchdf()
con.close()
print(f"papers_distinct rows: {len(papers):,}")

df = topics.merge(papers, on="pmid", how="left")
pct_missing_journal = df["journal"].isna().mean() * 100
pct_missing_year = df["year"].isna().mean() * 100
print(f"missing journal: {pct_missing_journal:.2f}%; missing year: {pct_missing_year:.2f}%")

df = df.merge(
    hier[["topic_id", "top_words", "size", "mid_level_id", "top_level_id"]],
    on="topic_id",
    how="left",
)
print("joined df shape:", df.shape)

topics rows: 89,230; unique topic_id: 150
hierarchy rows: 149
sweep shape: (9, 7)
papers_distinct rows: 121,908
missing journal: 0.00%; missing year: 0.00%
joined df shape: (89230, 10)


## 3. Intertopic distance map

In [3]:
try:
    from bertopic import BERTopic

    model = BERTopic.load(str(MODEL_DIR))
    fig = model.visualize_topics()
    fig.write_html(str(LANDSCAPE_HTML))
    print("wrote", LANDSCAPE_HTML)
except Exception as e:
    msg = f"Intertopic distance map unavailable: {type(e).__name__}: {e}"
    LANDSCAPE_HTML.write_text(msg + "\n")
    print(msg)
    print("wrote placeholder to", LANDSCAPE_HTML)

2026-05-21 23:33:44,459 - BERTopic - WARNING: You are loading a BERTopic model without explicitly defining an embedding model. If you want to also load in an embedding model, make sure to use `BERTopic.load(my_model, embedding_model=my_embedding_model)`.


wrote /Users/samersalman/Desktop/SciField/docs/figures/topic_landscape.html


## 4. Temporal heatmap of top-level topic share

In [4]:
df_nz = df[df["is_noise"] == False].copy()  # noqa: E712
df_nz = df_nz.dropna(subset=["year", "top_level_id"])
df_nz["year"] = df_nz["year"].astype(int)
df_nz["top_level_id"] = df_nz["top_level_id"].astype(int)

counts = df_nz.groupby(["top_level_id", "year"]).size().rename("n").reset_index()
pivot = counts.pivot(index="top_level_id", columns="year", values="n").fillna(0)
share = pivot.div(pivot.sum(axis=0), axis=1).fillna(0)

fig, ax = plt.subplots(figsize=(12, max(4, 0.4 * share.shape[0])))
im = ax.imshow(share.values, aspect="auto", cmap="viridis", vmin=0)
ax.set_xticks(range(share.shape[1]))
ax.set_xticklabels(share.columns, rotation=90, fontsize=8)
ax.set_yticks(range(share.shape[0]))
ax.set_yticklabels([f"T{tid}" for tid in share.index], fontsize=8)
ax.set_xlabel("Year")
ax.set_ylabel("Top-level topic")
ax.set_title("Top-level topic share by year (V1-S06)")
plt.colorbar(im, ax=ax, label="share within year")
fig.tight_layout()
fig.savefig(SHARE_BY_YEAR_PNG, dpi=120)
plt.close(fig)
print("wrote", SHARE_BY_YEAR_PNG)

wrote /Users/samersalman/Desktop/SciField/docs/figures/topic_share_by_year.png


## 5. Per-journal-per-year topic distribution (small multiples)

In [5]:
top_journals = df_nz["journal"].dropna().value_counts().head(10).index.tolist()
all_top_ids = sorted(df_nz["top_level_id"].unique().tolist())
cmap = plt.get_cmap("tab20", max(len(all_top_ids), 1))
colors = [cmap(i) for i in range(len(all_top_ids))]

fig, axes = plt.subplots(2, 5, figsize=(20, 8), sharex=False, sharey=True)
axes_flat = axes.flatten()
for ax, journal in zip(axes_flat, top_journals, strict=False):
    sub = df_nz[df_nz["journal"] == journal]
    if sub.empty:
        ax.set_title(f"{journal[:30]}\n(no data)", fontsize=8)
        ax.set_axis_off()
        continue
    c = sub.groupby(["top_level_id", "year"]).size().rename("n").reset_index()
    p = c.pivot(index="top_level_id", columns="year", values="n").fillna(0)
    p = p.reindex(index=all_top_ids, fill_value=0)
    s = p.div(p.sum(axis=0), axis=1).fillna(0)
    years = s.columns.values
    ax.stackplot(years, s.values, colors=colors, labels=[f"T{t}" for t in all_top_ids])
    ax.set_title(journal[:35], fontsize=9)
    ax.set_xlabel("Year", fontsize=8)
    ax.set_ylim(0, 1)
    ax.tick_params(axis="both", labelsize=7)

# Hide any unused axes if fewer than 10 journals.
for ax in axes_flat[len(top_journals) :]:
    ax.set_axis_off()

axes_flat[0].set_ylabel("Share", fontsize=8)
axes_flat[5].set_ylabel("Share", fontsize=8)
fig.suptitle("Top-level topic share by year, per journal (V1-S06)", fontsize=12)
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(SHARE_BY_JOURNAL_YEAR_PNG, dpi=120)
plt.close(fig)
print("wrote", SHARE_BY_JOURNAL_YEAR_PNG)

wrote /Users/samersalman/Desktop/SciField/docs/figures/topic_share_by_journal_year.png


## 6. 20-topic spot-check scaffold for Gate G1

Samer: fill the `clinical_interpretation` column in `docs/gates/G1_spotcheck.csv` — that completes Gate G1.

In [6]:
leaves = hier[hier["topic_id"] != -1].copy()
leaves["size_quartile"] = pd.qcut(
    leaves["size"], q=4, labels=["q1", "q2", "q3", "q4"], duplicates="drop"
)
sample = leaves.groupby("size_quartile", group_keys=False).apply(
    lambda g: g.nlargest(min(5, len(g)), "size")
)
if len(sample) > 20:
    sample = sample.head(20)


def _rep_titles(row):
    titles = []
    reps = row.get("representative_docs")
    if reps is not None:
        try:
            titles = [str(t) for t in list(reps) if t is not None][:3]
        except TypeError:
            titles = []
    if len(titles) < 3:
        sub = df[df["topic_id"] == row["topic_id"]]["title"].dropna().head(3 - len(titles))
        titles.extend(sub.tolist())
    while len(titles) < 3:
        titles.append("")
    return titles[:3]


rows_out = []
for _, row in sample.iterrows():
    t1, t2, t3 = _rep_titles(row)
    top_words = row.get("top_words")
    if top_words is None:
        words_str = ""
    else:
        try:
            words_str = " | ".join(str(w) for w in list(top_words))
        except TypeError:
            words_str = str(top_words)
    rows_out.append(
        {
            "topic_id": int(row["topic_id"]),
            "size": int(row["size"]),
            "top_words": words_str,
            "sample_title_1": t1,
            "sample_title_2": t2,
            "sample_title_3": t3,
            "clinical_interpretation": "",
        }
    )

spotcheck = pd.DataFrame(rows_out)
spotcheck.to_csv(SPOTCHECK_CSV, index=False)
print(f"wrote {len(spotcheck)} rows to {SPOTCHECK_CSV}")

wrote 20 rows to /Users/samersalman/Desktop/SciField/docs/gates/G1_spotcheck.csv
